# E. coli: local recombination and mutation rates, and hotspots

This notebook repeats the *E. coli* analysis of the paper (Section 3.4):

1. load the observed summary statistics of 836 alignment blocks of 27 *E. coli* genomes;
2. train NPE on 50,000 ClonalOrigin simulations on the *E. coli* clonal genealogy;
3. sample the posterior of $(\rho_s, \theta_s, \ell)$ for every block;
4. detect recombination and mutation hotspots with the ppDoG detector.

Run it from the `examples` folder. Training takes about 25 minutes on a laptop CPU.
The trained posterior is saved in `output/`, and later runs load it instead of training again.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from bacnpe.summary_stats import STAT_NAMES
from bacnpe.npe import LeaveLengthOut_NN, build_prior, train_npe, save_npe, load_npe, sample_posteriors
from bacnpe.detection import detect_hotspots, save_hotspot_tables

warnings.filterwarnings("ignore", category=UserWarning)

DATA = Path("data")
OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

## Load observed data

One row per alignment block. Positions refer to the reference genome *E. coli* 536.

In [ ]:
x_obs_info_table = pd.read_csv(DATA / "ecoli_block_info_updated.csv", index_col=0)
x_obs_info_table.head()

The observed statistics were computed in an older format with 46 columns.
Columns 16-31 are not used; dropping them leaves the 30 entries of `STAT_NAMES`
(29 summary statistics and the block length), as in the simulations.

In [ ]:
drop_col = range(16, 32)

x_obs = np.loadtxt(DATA / "ecoli_block_summary_stats.csv", delimiter=",")
x_obs = np.delete(x_obs, drop_col, axis=1)
print(x_obs.shape)
print("Lengths match the block table:", np.allclose(x_obs[:, -1], x_obs_info_table["Gene_Length"]))
pd.DataFrame(x_obs, columns=STAT_NAMES).head()

### Delete observations with no signal

Blocks without segregating sites carry almost no information about the rates, and blocks
longer than 10,000 bp are outside the prior on $\ell$. (There are none of either in this dataset.)

In [ ]:
no_signal = x_obs[:, STAT_NAMES.index("prop_segregating_sites")] == 0
too_long = x_obs_info_table["Gene_Length"].to_numpy() > 10_000
keep = ~(no_signal | too_long)
print(f"No signal: {no_signal.sum()}, too long: {too_long.sum()}, kept: {keep.sum()} of {len(keep)}")

x_obs = x_obs[keep]
gene_info = x_obs_info_table[keep]

## Load simulation data

50,000 simulations on the *E. coli* clonal genealogy with $\delta = 542$ bp, using the priors
$\rho_s \sim U(0, 0.2)$, $\theta_s \sim U(0, 0.1)$ and $\ell \sim$ DiscreteUniform$\{400, \dots, 10000\}$.
`ecoli_theta.csv` holds the parameters $(\rho_s, \theta_s, \ell)$ and `ecoli_x.csv` the 30 statistics,
row by row. They can be generated with `bacnpe.parallel_sim` (see `01_simulate.py`).

In [ ]:
theta = np.loadtxt(DATA / "ecoli_theta.csv", delimiter=",")
x = np.loadtxt(DATA / "ecoli_x.csv", delimiter=",")
print(theta.shape, x.shape)
print("Parameter ranges:", theta.min(axis=0), theta.max(axis=0))

## NPE

### Create prior to pass range knowledge to NPE

Use the ranges of the simulations. The posterior then discards draws outside these ranges.

In [ ]:
prior = build_prior(rho_range=(0.0, 0.2), theta_range=(0.0, 0.1), L_range=(400, 10000))
prior

### Using all simulations

The configuration selected in the paper (Table S3):

- compression network: one hidden layer of 128 units, 2 learned features plus $\ell$;
- neural spline flow: 8 transforms, 30 hidden features, 5 bins;
- training: learning rate $10^{-4}$, batch size 100, early stopping after 20 epochs without improvement.

Simulations with NaN statistics are dropped before training.

In [ ]:
seed = 100
save_path = OUTPUT / "ecoli_trained_npe.pt"

if save_path.exists():
    posterior = load_npe(save_path)["posterior"]
    print(f"Loaded the trained posterior from {save_path}")
else:
    torch.manual_seed(seed)                      # reproducible embedding weights
    embedding_net = LeaveLengthOut_NN(input_dim=30, num_hiddens=128, num_hidden_layers=1, num_outputs=2)
    result = train_npe(
        theta, x, prior,
        seed=seed,
        embedding_net=embedding_net,
        model="nsf", hidden_features=30, num_transforms=8, num_bins=5,
        training_batch_size=100, learning_rate=1e-4,
        max_num_epochs=500, stop_after_epochs=20, clip_max_norm=None,
    )
    print(f"Dropped {result.settings['n_dropped']} simulations with NaN/inf statistics")
    save_npe(result, save_path)
    posterior = result.posterior

### Training and validation loss

Only available right after training (not when the posterior was loaded from file).

In [ ]:
if "result" in globals():
    summary = result.inference.summary
    plt.figure(figsize=(8, 4))
    plt.plot(summary["training_loss"], label="Training loss")
    plt.plot(summary["validation_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()
    plt.show()

### Plug in observation data to get posterior

`theta_post[g, m]` is posterior draw $m$ of $(\rho_s, \theta_s, \ell)$ for block $g$.

In [ ]:
num_posterior_samples = 1000
theta_post = sample_posteriors(posterior, x_obs, num_samples=num_posterior_samples, seed=seed)
theta_post.shape

As a check of the inference (Figure S7 of the paper), the inferred block lengths should match the observed ones.

In [ ]:
L_obs = x_obs[:, -1]
L_q = np.quantile(theta_post[:, :, 2], [0.025, 0.5, 0.975], axis=1)

plt.figure(figsize=(5, 5))
plt.errorbar(L_obs, L_q[1], yerr=[L_q[1] - L_q[0], L_q[2] - L_q[1]], fmt="o", ms=2, alpha=0.5,
             label="Posterior median ± 95% CI")
plt.plot([L_obs.min(), L_obs.max()], [L_obs.min(), L_obs.max()], "r--", lw=1)
plt.xlabel("True length (bp)")
plt.ylabel("Estimated length (bp)")
plt.legend()
plt.show()

## Detect hotspots

ppDoG settings of the paper for *E. coli* (Table S1): narrow bandwidth 100 kb, broad bandwidth 1 Mb,
$\tau = 0.2$, probability threshold 0.9, and minimum fold enrichment 4 (recombination)
and 100 (mutation). Blocks are placed at their start positions on the 4,938,906 bp chromosome.

In [ ]:
gene_starts = gene_info["Start_pos_raw"].to_numpy()
gene_ends = gene_info["End_pos_raw"].to_numpy()
chromosome_length_bp = 4_938_906

settings = dict(
    gene_ends=gene_ends,
    narrow_bandwidth_bp=100_000,
    broad_bandwidth_bp=1_000_000,
    probability_threshold=0.90,
    residual_log_sd=0.20,
    random_state=10,
    max_gap_bp=800_000,
)

### Recombination hotspots

In [ ]:
recomb = detect_hotspots(gene_starts, theta_post[:, :, 0], chromosome_length_bp,
                         min_fold_enrichment=4, **settings)
print(recomb.summary())

recomb.plot(
    rate_label="Recombination rate",
    title_text="E. coli Recombination Rate with Detected Candidate Hotspots",
    hotspot_padding_bp=10_000,
    save_path=OUTPUT / "ecoli_local_recomb_dog.pdf",
)
plt.show()

In [ ]:
recomb_hotspots = recomb.to_dataframe()
recomb_hotspots

Blocks in the first recombination hotspot:

In [ ]:
segments = recomb.segment_gene_indices()
gene_info.iloc[segments[0]] if segments else "No hotspot detected"

### Mutation hotspots

In [ ]:
mutate = detect_hotspots(gene_starts, theta_post[:, :, 1], chromosome_length_bp,
                         min_fold_enrichment=100.0, **settings)
print(mutate.summary())

mutate.plot(
    rate_label="Mutation rate",
    title_text="E. coli Mutation Rate with Detected Candidate Hotspots",
    hotspot_padding_bp=10_000,
    save_path=OUTPUT / "ecoli_local_mutate_dog.pdf",
)
plt.show()

In [ ]:
mutate_hotspots = mutate.to_dataframe()
mutate_hotspots

### Save the hotspot tables

One sheet per rate, as Table S6 of the paper.

In [ ]:
save_hotspot_tables({"Recombination": recomb_hotspots, "Mutation": mutate_hotspots},
                    OUTPUT / "ecoli_hotspots.xlsx", overwrite=True)